# Agentic RAG on HotpotQA



- Dataset: `hotpotqa/hotpot_qa`, `distractor`, `validation`
- Retriever: OpenAI Embedding + cosine similarity
- LLM: `gpt-4o`
- Embedding model: `text-embedding-3-small`
- Evaluation: Exact Match (EM) / token-level F1
-：Agentic retrieval loop、query rewrite、


 RAG：

`Question -> Retrieve once -> Answer`

Agentic RAG：

`Question -> Retrieve -> Agent decides SEARCH or ANSWER -> Retrieve again if needed -> ... -> Answer`



In [ ]:
# Cell 1：安装依赖
%pip install -q openai datasets pandas tqdm numpy

In [ ]:
# Cell 2：初始化
import os
import re
import json
import string
from collections import Counter
from getpass import getpass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from openai import OpenAI


if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

CHAT_MODEL = "gpt-4o"
EMBEDDING_MODEL = "text-embedding-3-small"

NUM_SAMPLES = 200


TOP_K_PER_STEP = 2
MAX_RETRIEVAL_STEPS = 3
MAX_EVIDENCE_DOCS = 6

dataset = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split="validation"
)

demo_data = dataset.shuffle(seed=42).select(range(NUM_SAMPLES))

print("samples:", len(demo_data))
print("chat model:", CHAT_MODEL)
print("embedding model:", EMBEDDING_MODEL)

OPENAI_API_KEY: ··········


Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

samples: 200
chat model: gpt-4o
embedding model: text-embedding-3-small


In [ ]:
# Cell 3：将每个候选段落转成文档块
def build_documents(context):
    documents = []

    for doc_id, (title, sentences) in enumerate(
        zip(context["title"], context["sentences"])
    ):
        text = " ".join(sentences).strip()

        if text:
            documents.append({
                "doc_id": doc_id,
                "title": title,
                "text": text,
                "content": f"Title: {title}\nContent: {text}",
            })

    return documents


documents = build_documents(demo_data[0]["context"])
print(documents[0]["content"][:500])

Title: Robin Barton
Content: Robin Barton (born 5 November 1958) is a British art dealer dealing primarily with Banksy's.  Barton studied photography and graphic design at the Exeter College of Art and Design and this was his first encounter with Russell Young.  Moving to London in 1980 he began working as a freelance photographer for music and fashion publications "Sounds", "NME", "Blitz", "The Face" moving on to working regularly for pioneering "Independent Magazine" photographing amongst othe


In [ ]:
# Cell 4：Embedding、余弦相似度与可复用检索索引

def get_embeddings(texts):
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    return np.array(
        [item.embedding for item in response.data],
        dtype=np.float32
    )


def normalize_vectors(vectors):
    norms = np.linalg.norm(vectors, axis=-1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return vectors / norms


def prepare_retrieval_index(documents):
    """
    每个 HotpotQA 样本只对候选文档做一次 embedding。
    Agent 后续多轮 SEARCH 时只需要重新计算 query embedding。
    """
    document_embeddings = get_embeddings(
        [doc["content"] for doc in documents]
    )
    return normalize_vectors(document_embeddings)


def retrieve(
    query,
    documents,
    document_embeddings,
    top_k=2,
    exclude_doc_ids=None,
):
    """
    在当前样本的候选段落中检索。

    exclude_doc_ids:
        已经收集过的 doc_id。默认不重复返回，促使 Agent 探索新的证据。
    """
    exclude_doc_ids = set(exclude_doc_ids or [])

    query_embedding = get_embeddings([query])[0]
    query_embedding = normalize_vectors(query_embedding.reshape(1, -1))[0]

    scores = document_embeddings @ query_embedding
    ranked_indices = np.argsort(scores)[::-1]

    retrieved_docs = []

    for idx in ranked_indices:
        doc = documents[idx]

        if doc["doc_id"] in exclude_doc_ids:
            continue

        result = doc.copy()
        result["score"] = float(scores[idx])
        retrieved_docs.append(result)

        if len(retrieved_docs) >= top_k:
            break

    return retrieved_docs

In [ ]:
# Cell 5：Agent Controller
#
# Agent 每轮只做一个高层决策：
# 1) SEARCH：当前证据不足，生成下一条检索 query
# 2) ANSWER：当前证据已经足够，直接给最终答案
#
# 注意：这里不要求模型输出 chain-of-thought，只要求结构化 action。

def format_evidence(evidence_docs):
    if not evidence_docs:
        return "(No evidence collected yet.)"

    blocks = []

    for i, doc in enumerate(evidence_docs, start=1):
        blocks.append(
            f"[Evidence {i}]\n"
            f"Title: {doc['title']}\n"
            f"Content: {doc['text']}"
        )

    return "\n\n".join(blocks)


def agent_decide(
    question,
    evidence_docs,
    search_history,
    current_step,
    max_steps,
):
    evidence_text = format_evidence(evidence_docs)
    history_text = (
        "\n".join(f"- {q}" for q in search_history)
        if search_history
        else "(none)"
    )

    prompt = f"""You are the controller of an Agentic RAG system for multi-hop QA.

You must use ONLY the collected evidence.
Do not use outside knowledge.

Your task is to choose exactly one action:

1. SEARCH
Choose SEARCH when the evidence is not yet sufficient to answer.
Generate a concise search query that targets the missing entity, relation,
bridge fact, comparison, date, location, or other missing information.

2. ANSWER
Choose ANSWER only when the evidence is sufficient to determine the answer.
The answer must be short and contain only the final answer.

Important rules:
- Multi-hop questions often require evidence from two different passages.
- Do not repeat an earlier search query unless absolutely necessary.
- Do not invent facts.
- Do not output chain-of-thought.
- Return valid JSON only.

JSON schema for SEARCH:
{{
  "action": "search",
  "query": "next search query"
}}

JSON schema for ANSWER:
{{
  "action": "answer",
  "answer": "short final answer"
}}

Question:
{question}

Collected Evidence:
{evidence_text}

Previous Search Queries:
{history_text}

Retrieval step:
{current_step} / {max_steps}
"""

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You control retrieval for a multi-hop QA system. "
                    "Use only supplied evidence and return JSON only."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=120,
    )

    raw = response.choices[0].message.content.strip()

    try:
        decision = json.loads(raw)
    except json.JSONDecodeError:
        # 保守 fallback：无法解析时继续检索，而不是凭空回答
        decision = {
            "action": "search",
            "query": question,
        }

    action = str(decision.get("action", "")).lower().strip()

    if action == "answer":
        answer = str(decision.get("answer", "")).strip()
        if answer:
            return {
                "action": "answer",
                "answer": answer,
            }

    query = str(decision.get("query", "")).strip()

    if not query:
        query = question

    return {
        "action": "search",
        "query": query,
    }


def force_final_answer(question, evidence_docs):
    """
    达到最大检索步数后，用已经收集到的证据生成最终答案。
    """
    evidence_text = format_evidence(evidence_docs)

    prompt = f"""You are a careful multi-hop question-answering assistant.

Answer using ONLY the evidence below.
Do not use outside knowledge.
Return only a short final answer, without explanation.

If the evidence is insufficient, return exactly:
Cannot be determined from the provided context.

Question:
{question}

Evidence:
{evidence_text}
"""

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer strictly from the supplied evidence. "
                    "Do not add outside knowledge."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=100,
    )

    return response.choices[0].message.content.strip()

In [ ]:
# Cell 6：Agentic RAG 主循环

def agentic_rag_answer(
    question,
    documents,
    top_k_per_step=TOP_K_PER_STEP,
    max_steps=MAX_RETRIEVAL_STEPS,
    max_evidence_docs=MAX_EVIDENCE_DOCS,
):
    """
    简化版 Agentic RAG：

    1. 用原始 question 做第一轮检索
    2. Agent 查看累计 evidence
    3. Agent 决定：
       - ANSWER：结束
       - SEARCH(query)：改写 query 并继续检索
    4. 最多 max_steps 轮检索
    """

    document_embeddings = prepare_retrieval_index(documents)

    evidence_docs = []
    collected_doc_ids = set()
    search_history = []
    trace = []

    # 第一轮从原问题开始，便于与普通 RAG 对齐
    current_query = question

    for step in range(1, max_steps + 1):
        search_history.append(current_query)

        new_docs = retrieve(
            query=current_query,
            documents=documents,
            document_embeddings=document_embeddings,
            top_k=top_k_per_step,
            exclude_doc_ids=collected_doc_ids,
        )

        for doc in new_docs:
            if doc["doc_id"] not in collected_doc_ids:
                evidence_docs.append(doc)
                collected_doc_ids.add(doc["doc_id"])

        # 控制累计上下文长度
        evidence_docs = evidence_docs[:max_evidence_docs]

        decision = agent_decide(
            question=question,
            evidence_docs=evidence_docs,
            search_history=search_history,
            current_step=step,
            max_steps=max_steps,
        )

        step_trace = {
            "step": step,
            "search_query": current_query,
            "retrieved_titles": [doc["title"] for doc in new_docs],
            "retrieval_scores": [
                round(doc["score"], 4) for doc in new_docs
            ],
            "agent_action": decision["action"],
        }

        if decision["action"] == "answer":
            step_trace["final_answer"] = decision["answer"]
            trace.append(step_trace)

            return (
                decision["answer"],
                evidence_docs,
                trace,
                "agent_answer",
            )

        next_query = decision["query"]
        step_trace["next_query"] = next_query
        trace.append(step_trace)

        # 如果已经没有新的文档可检索，提前结束并基于现有证据回答
        if not new_docs:
            break

        # 防止完全相同的 query 无限重复
        if next_query.strip().lower() in {
            q.strip().lower() for q in search_history
        }:
            next_query = (
                question
                + " missing bridge entity or supporting fact"
            )

        current_query = next_query

    final_answer = force_final_answer(
        question=question,
        evidence_docs=evidence_docs,
    )

    return (
        final_answer,
        evidence_docs,
        trace,
        "max_steps_or_no_new_docs",
    )

In [ ]:
# Cell 7：先运行一个样本，查看 Agent 的检索轨迹

item = demo_data[0]
documents = build_documents(item["context"])

prediction, evidence_docs, trace, stop_reason = agentic_rag_answer(
    question=item["question"],
    documents=documents,
)

print("Question:", item["question"])
print("Gold:", item["answer"])
print("Prediction:", prediction)
print("Stop reason:", stop_reason)
print()

for step in trace:
    print(
        f"Step {step['step']} | "
        f"query={step['search_query']!r} | "
        f"retrieved={step['retrieved_titles']} | "
        f"action={step['agent_action']}"
    )

    if "next_query" in step:
        print("  next query:", step["next_query"])

Question: What nationality was Oliver Reed's character in the film Royal Flash?
Gold: Prussian
Prediction: German
Stop reason: agent_answer

Step 1 | query="What nationality was Oliver Reed's character in the film Royal Flash?" | retrieved=['Oliver Reed', 'Royal Flash (film)'] | action=answer


In [ ]:
# Cell 8：EM / F1 评测函数（与基础版本保持一致）

def normalize_answer(text):
    text = text.lower()
    text = "".join(
        ch for ch in text
        if ch not in string.punctuation
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match(prediction, gold):
    return int(
        normalize_answer(prediction)
        == normalize_answer(gold)
    )


def token_f1(prediction, gold):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()

    common = Counter(pred_tokens) & Counter(gold_tokens)
    overlap = sum(common.values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)

    return (
        2 * precision * recall
        / (precision + recall)
    )

In [ ]:
# Cell 9：Supporting-title retrieval 指标
#
# HotpotQA 除了 answer，还提供 supporting_facts。
# 对 Agentic RAG 来说，检查是否真的检索到了支持答案的文档很有价值。

def get_gold_supporting_titles(item):
    supporting_facts = item["supporting_facts"]

    # Hugging Face HotpotQA 一般为：
    # {"title": [...], "sent_id": [...]}
    if isinstance(supporting_facts, dict):
        return list(dict.fromkeys(supporting_facts["title"]))

    # 兼容其他可能表示
    titles = []
    for fact in supporting_facts:
        if isinstance(fact, (list, tuple)) and len(fact) >= 1:
            titles.append(fact[0])

    return list(dict.fromkeys(titles))


def supporting_title_recall(retrieved_titles, gold_titles):
    gold = set(gold_titles)
    retrieved = set(retrieved_titles)

    if not gold:
        return 0.0

    return len(gold & retrieved) / len(gold)


def all_supporting_titles_retrieved(retrieved_titles, gold_titles):
    gold = set(gold_titles)
    retrieved = set(retrieved_titles)

    if not gold:
        return 0

    return int(gold.issubset(retrieved))

In [ ]:
# Cell 10：在 HotpotQA 上批量运行 Agentic RAG
#
# 注意：Agentic RAG 每个样本可能调用多次 Chat Completion，
# API 成本和运行时间会高于一次检索的普通 RAG。
# 第一次建议先：
# demo_eval_data = demo_data.select(range(20))
#
# 确认流程无误后，再改回：
# demo_eval_data = demo_data

demo_eval_data = demo_data

agentic_results = []

for item in tqdm(
    demo_eval_data,
    desc="Agentic RAG + GPT-4o"
):
    documents = build_documents(item["context"])

    prediction, evidence_docs, trace, stop_reason = (
        agentic_rag_answer(
            question=item["question"],
            documents=documents,
            top_k_per_step=TOP_K_PER_STEP,
            max_steps=MAX_RETRIEVAL_STEPS,
        )
    )

    retrieved_titles = [
        doc["title"] for doc in evidence_docs
    ]

    gold_supporting_titles = get_gold_supporting_titles(item)

    agentic_results.append({
        "id": item["id"],
        "question": item["question"],
        "gold_answer": item["answer"],
        "prediction": prediction,
        "retrieved_titles": retrieved_titles,
        "gold_supporting_titles": gold_supporting_titles,
        "supporting_title_recall": supporting_title_recall(
            retrieved_titles,
            gold_supporting_titles,
        ),
        "all_supporting_titles_retrieved": (
            all_supporting_titles_retrieved(
                retrieved_titles,
                gold_supporting_titles,
            )
        ),
        "num_retrieval_steps": len(trace),
        "search_queries": [
            step["search_query"] for step in trace
        ],
        "stop_reason": stop_reason,
        "trace": trace,
    })

agentic_df = pd.DataFrame(agentic_results)

agentic_df["EM"] = agentic_df.apply(
    lambda x: exact_match(
        x["prediction"],
        x["gold_answer"],
    ),
    axis=1,
)

agentic_df["F1"] = agentic_df.apply(
    lambda x: token_f1(
        x["prediction"],
        x["gold_answer"],
    ),
    axis=1,
)

display(
    agentic_df[
        [
            "question",
            "gold_answer",
            "prediction",
            "retrieved_titles",
            "num_retrieval_steps",
            "EM",
            "F1",
        ]
    ].head(20)
)

print(f"Agentic RAG average EM: {agentic_df['EM'].mean():.3f}")
print(f"Agentic RAG average F1: {agentic_df['F1'].mean():.3f}")
print(
    "Supporting-title recall: "
    f"{agentic_df['supporting_title_recall'].mean():.3f}"
)
print(
    "All supporting titles retrieved: "
    f"{agentic_df['all_supporting_titles_retrieved'].mean():.3f}"
)
print(
    "Average retrieval steps: "
    f"{agentic_df['num_retrieval_steps'].mean():.2f}"
)

Agentic RAG + GPT-4o:   0%|          | 0/200 [00:00<?, ?it/s]

,question,gold_answer,prediction,retrieved_titles,num_retrieval_steps,EM,F1
0,What nationality was Oliver Reed's character i...,Prussian,Prussian,"[Oliver Reed, Royal Flash (film), Otto von Bis...",2,1,1.000000
1,Pacific Mozart Ensemble performed which German...,Kurt Julian Weill,Kurt Weill,"[The Flight Across the Ocean, Pacific Mozart E...",1,0,0.800000
2,"Who released the song ""With or Without You"" fi...",U2,U2,"[With or Without You, Believe (Jai McDowall al...",1,1,1.000000
3,"What Kentucky county has a population of 60,31...",Oldham County,Oldham County,"[Lake Louisvilla, Louisville, Oldham County, K...",1,1,1.000000
4,"Para Hills West, South Australia lies within a...","138,535","138,535","[Para Hills West, South Australia, Para Hills,...",2,1,1.000000
5,"In what year was the narrator of ""Blackadder's...",1959,1959,"[Blackadder's Christmas Carol, A Christmas Car...",2,1,1.000000
6,Thomas Fitch defended John Henry Holliday when...,"October 25, 1881","October 26, 1881","[Thomas Fitch (politician), O.K. Corral hearin...",1,0,0.666667
7,The actor that played Gutthi on Comedy nights ...,Vaisakhi List,Cannot be determined from the provided context.,"[Smeep Kang, Bhaji in Problem, Sunil Grover, K...",3,0,0.000000
8,"Victor Varnada directed Twisted Fortune, which...",Charlie Murphy,Charlie Murphy,"[Twisted Fortune, Eddie Murphy Raw]",1,1,1.000000
9,After his curacy at the village that is a subu...,Dudman,Bill Dudman,"[John Gathercole, Bill Dudman]",1,0,0.666667


Agentic RAG average EM: 0.600
Agentic RAG average F1: 0.726
Supporting-title recall: 0.855
All supporting titles retrieved: 0.730
Average retrieval steps: 1.49


In [ ]:
# Cell 11：查看 Agent 的具体行为

def show_agent_trace(row_index):
    row = agentic_df.iloc[row_index]

    print("=" * 80)
    print("Question:", row["question"])
    print("Gold:", row["gold_answer"])
    print("Prediction:", row["prediction"])
    print("EM:", row["EM"])
    print("F1:", round(row["F1"], 3))
    print("Gold supporting titles:", row["gold_supporting_titles"])
    print("Retrieved titles:", row["retrieved_titles"])
    print("Stop reason:", row["stop_reason"])
    print()

    for step in row["trace"]:
        print(f"[Step {step['step']}]")
        print("Search query:", step["search_query"])
        print("Retrieved:", step["retrieved_titles"])
        print("Scores:", step["retrieval_scores"])
        print("Agent action:", step["agent_action"])

        if "next_query" in step:
            print("Next query:", step["next_query"])

        if "final_answer" in step:
            print("Final answer:", step["final_answer"])

        print()


# 示例
show_agent_trace(0)

Question: What nationality was Oliver Reed's character in the film Royal Flash?
Gold: Prussian
Prediction: Prussian
EM: 1
F1: 1.0
Gold supporting titles: ['Royal Flash (film)', 'Otto von Bismarck']
Retrieved titles: ['Oliver Reed', 'Royal Flash (film)', 'Otto von Bismarck', 'The Duke of Hamilton']
Stop reason: agent_answer

[Step 1]
Search query: What nationality was Oliver Reed's character in the film Royal Flash?
Retrieved: ['Oliver Reed', 'Royal Flash (film)']
Scores: [0.6285, 0.6245]
Agent action: search
Next query: Otto von Bismarck nationality

[Step 2]
Search query: Otto von Bismarck nationality
Retrieved: ['Otto von Bismarck', 'The Duke of Hamilton']
Scores: [0.732, 0.1283]
Agent action: answer
Final answer: Prussian



In [ ]:
# Cell 12：可选——与 Agentic RAG 使用同一检索器的 Vanilla RAG baseline
#
# Vanilla RAG: Question -> Retrieve once -> Answer
# Agentic RAG: Question -> Retrieve -> Decide -> Search again / Answer

def vanilla_rag_answer(question, documents, top_k=3):
    document_embeddings = prepare_retrieval_index(documents)

    retrieved_docs = retrieve(
        query=question,
        documents=documents,
        document_embeddings=document_embeddings,
        top_k=top_k,
    )

    retrieved_context = format_evidence(retrieved_docs)

    prompt = f"""You are a careful multi-hop question-answering assistant.

Answer the question using only the retrieved evidence below.
Do not use outside knowledge.
Return only a short final answer, without explaining your reasoning.

If the evidence is insufficient, return exactly:
Cannot be determined from the provided context.

Question:
{question}

Retrieved Evidence:
{retrieved_context}
"""

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer strictly using the retrieved evidence. "
                    "Do not add outside knowledge."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=100,
    )

    return (
        response.choices[0].message.content.strip(),
        retrieved_docs,
    )